In [ ]:
import bisect
import csv
import concurrent.futures
import datetime
import faiss
import glob
import gc
import json
import math
import multiprocessing
import os
import pickle
import random
import re
import string
import subprocess
import time
import torch
import uuid
import yaml

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import multiprocessing as mp
import polars as pl
import torch.nn as nn
import torch.optim as optim

from bisect import bisect_left
from collections import defaultdict, Counter
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime, timedelta
from tqdm import tqdm
from pathlib import Path
from multiprocessing import Pool, cpu_count
from functools import partial
from typing import Set, Tuple, Dict

In [ ]:
# Disk model used as the running example.
DISK_MODEL_ID = "ST8000NM0055"

# Backward-compatible alias used by later cells.
model_serial_id = DISK_MODEL_ID

# All paths are relative to the repository root.
PROJECT_ROOT = Path.cwd().resolve()

INPUT_ROOT = PROJECT_ROOT / "input"
BACKBLAZE_INPUT_DIR = INPUT_ROOT / "backblaze"
BACKBLAZE_SOURCE_DIR = BACKBLAZE_INPUT_DIR / "original_csv"

SMART_ATTRIBUTE_PATH = INPUT_ROOT / "smart_attributes.csv"
DISK_INFO_PATH = BACKBLAZE_INPUT_DIR / "disk_info.csv"

OUTPUT_ROOT = PROJECT_ROOT / "output"
OUTPUT_DATA_DIR = OUTPUT_ROOT / "data" / DISK_MODEL_ID
TEXTUAL_OUTPUT_DIR = OUTPUT_DATA_DIR / "monthly_textual"
MODEL_OUTPUT_DIR = OUTPUT_ROOT / "models" / DISK_MODEL_ID
RESULT_OUTPUT_PATH = OUTPUT_ROOT / "results" / f"{DISK_MODEL_ID}.csv"

OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
TEXTUAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Disk model: {DISK_MODEL_ID}")
print(f"Input directory: {BACKBLAZE_SOURCE_DIR}")
print(f"Output data directory: {OUTPUT_DATA_DIR}")

## Load Data

In [ ]:
df_smart_des = pd.read_csv(SMART_ATTRIBUTE_PATH)
disk_info = pd.read_csv(DISK_INFO_PATH)
print(df_smart_des.columns)

In [ ]:
df_smart_des['threshold'].unique()

In [ ]:
# Obtain the data format from the first available Backblaze CSV file.
source_files = sorted(BACKBLAZE_SOURCE_DIR.glob("*.csv"))
if not source_files:
    raise FileNotFoundError(f"No Backblaze source CSV files found under: {BACKBLAZE_SOURCE_DIR}")

df_temp = pd.read_csv(source_files[0], nrows=5)

feature_columns = [col for col in df_temp.columns if col.endswith("_raw")]
basic_columns = [
    "date",
    "serial_number",
    "model",
    "capacity_bytes",
    "failure",
    "datacenter",
    "cluster_id",
    "vault_id",
    "pod_id",
    "pod_slot_num",
    "is_legacy_format",
]

In [ ]:
def process_file(filepath, model_id, usecols_list, chunksize=100_000):
    collected = []

    dtype_dict = {
        5: 'str',
        'datacenter': 'str'
    }

    try:
        for chunk in pd.read_csv(
            filepath,
            usecols=usecols_list,
            chunksize=chunksize,
            dtype=dtype_dict,
            low_memory=False,
            on_bad_lines='warn',
        ):
            filtered = chunk[chunk['model'] == model_id]
            if not filtered.empty:
                collected.append(filtered)

        if not collected:
            return None

        # Merge once per source file to avoid keeping many tiny DataFrames alive
        return pd.concat(collected, ignore_index=True)

    except Exception as e:
        print(f"Error processing {os.path.basename(filepath)}: {e}")
        return None

In [ ]:
files = sorted(BACKBLAZE_SOURCE_DIR.glob("*.csv"))

process_partial = partial(
    process_file,
    model_id=model_serial_id,
    usecols_list=basic_columns + feature_columns,
    chunksize=100_000
)

n_processes = max(1, cpu_count() - 1)
print(f"Using {n_processes} processes")

with Pool(processes=n_processes) as pool:
    results = [result for result in pool.map(process_partial, files) if result is not None]

if results:
    print(f"Concatenating {len(results):,} filtered file-level DataFrames...")
    df_final = pd.concat(results, ignore_index=True)
    print(f"Final shape: {df_final.shape}")
else:
    df_final = pd.DataFrame()
    print("No matching rows found across all files")

In [ ]:
df = df_final.dropna(axis=1, how='all')
del df_final
df['date'] = pd.to_datetime(df['date'], format='mixed')
df = df.reset_index()
df = df.drop(["index"], axis=1)
print(df.shape)

In [ ]:
failed_serials = df[df['failure'] == 1]['serial_number'].unique()
print(f"Number of serials with at least one failure: {len(failed_serials):,}")

all_serials = df['serial_number'].unique()
healthy_serials = np.setdiff1d(all_serials, failed_serials)
print(f"Number of serials that never failed: {len(healthy_serials):,}")

n_sample = int(len(healthy_serials) * (1/2))
sampled_healthy = np.random.choice(healthy_serials, size=n_sample, replace=False)
print(f"Selected {len(sampled_healthy):,} healthy serials (≈1/2)")

keep_serials = np.concatenate([failed_serials, sampled_healthy])
keep_serials_set = set(keep_serials)
df_sampled = df[df['serial_number'].isin(keep_serials_set)].copy()

print(f"Original rows  : {len(df):,}")
print(f"After sampling : {len(df_sampled):,}")
print(f"Reduction      : {len(df) - len(df_sampled):,} rows removed")
print(f"Remaining serials: {df_sampled['serial_number'].nunique():,}")

In [ ]:
df = df_sampled
del df_sampled
gc.collect()

## Build Metadata

In [ ]:
d2t_start_time = time.perf_counter()

In [ ]:
threshold_rules = {}
related_map = {}

for _, row in df_smart_des.iterrows():
    name = row['name']
    thresh = row['threshold']
    threshold_rules[name] = thresh
    
    rel_str = row.get('related', '')
    if pd.isna(rel_str) or rel_str in ('—', '', '-'):
        related_list = []
    else:
        related_list = [x.strip() for x in str(rel_str).split(',')]
    related_map[name] = set(related_list)

In [ ]:
pct_attrs = set()
for attr, rule in threshold_rules.items():
    if rule.startswith(('upper_pct:', 'lower_pct:')):
        pct_attrs.add(attr)

quantiles = {}
for col in pct_attrs:
    if col in df.columns and df[col].dtype in ['int64', 'float64']:
        q_vals = df[col].quantile([0.01, 0.05, 0.10, 0.25, 0.75, 0.90, 0.95, 0.99])
        quantiles[col] = {
            'q01': q_vals[0.01],
            'q05': q_vals[0.05],
            'q10': q_vals[0.10],
            'q25': q_vals[0.25],
            'q75': q_vals[0.75],
            'q90': q_vals[0.90],
            'q95': q_vals[0.95],
            'q99': q_vals[0.99],
        }
    else:
        quantiles[col] = None

## Helper Functions

In [ ]:
def get_description(value, attr_name, threshold_rules, quantiles):
    if pd.isna(value):
        return 'N/A'
    
    rule = threshold_rules.get(attr_name, 'info')
    
    if rule in ('info', 'ignore', 'trend', 'model_specific'):
        return 'N/A'
    
    if rule == 'hard_zero':
        if value == 0:
            return 'normal'
        elif value <= 2:
            return 'high'
        else:
            return 'extremely high'

    if rule == 'soft_zero':
        if value == 0:
            return 'normal'
        elif value <= 5:
            return 'slightly high'
        elif value <= 30:
            return 'high'
        else:
            return 'extremely high'

    if rule.startswith('upper_pct:'):
        try:
            pct = int(rule.split(':', 1)[1])
            q_key = f'q{pct:02d}'
            q_val = quantiles.get(attr_name, {}).get(q_key)
            if q_val is None:
                return 'N/A'
            
            if value > q_val:
                if pct >= 99:
                    return 'extremely high'
                elif pct >= 95:
                    return 'high'
                else:
                    return 'slightly high'
            return 'normal'
        except Exception:
            return 'N/A'
    
    if rule.startswith('lower_pct:'):
        try:
            pct = int(rule.split(':', 1)[1])
            q_key = f'q{pct:02d}'
            q_val = quantiles.get(attr_name, {}).get(q_key)
            if q_val is None:
                return 'N/A'
            
            if value < q_val:
                if pct <= 1 or pct <= 5:
                    return 'extremely low' if pct <= 1 else 'low'
                else:
                    return 'slightly low'
            return 'normal'
        except Exception:
            return 'N/A'
    
    if rule.startswith('absolute:'):
        try:
            limit = float(rule.split(':', 1)[1])
            if value > limit + 15:
                return 'extremely high'
            elif value > limit + 5:
                return 'high'
            elif value > limit:
                return 'slightly high'
            else:
                return 'normal'
        except Exception:
            return 'N/A'
    
    return 'N/A'

In [ ]:
def compute_upcoming_vectorized(group):
    n = len(group)
    if n < 2:
        return ["No"] * n
    
    dates = group['date']
    failures = group['failure']
    
    s = dates.where(failures == 1, pd.NaT)
    next_incl = s.bfill()
    next_strict = next_incl.mask(failures == 1, pd.NaT)
    next_fail_dates = next_strict.bfill()

    has_upcoming = next_fail_dates.notna() & (next_fail_dates <= dates + timedelta(days=7))
    labels = np.full(n, "No", dtype=object)
    yes_mask = has_upcoming.values
    if yes_mask.any():
        yes_dates_str = next_fail_dates[yes_mask].dt.strftime('%Y/%m/%d').values
        yes_labels = np.array([f"Yes (on {d})" for d in yes_dates_str], dtype=object)
        labels[yes_mask] = yes_labels
    
    return labels.tolist()

In [ ]:
def has_recent_failure(levels, date, delta):
    if not levels:
        return False, None
    for level_name, failure_dates in levels:
        if not failure_dates:
            continue
        low = date - delta
        i = bisect.bisect_left(failure_dates, low)
        if i < len(failure_dates) and failure_dates[i] <= date:
            return True, level_name
    return False, None


def query_disk_info(model_name, info):
    global disk_info
    if model_name in disk_info['ModelName'].values:
        r = disk_info[disk_info['ModelName'] == model_name].iloc[0]
        return r.get(info, "")
    return ""

In [ ]:
def compute_recent_trend_vectorized(
    group,
    attr='smart_195_raw',
    lookback_days=7,
    min_points=3,
    strict_rel_change=0.10,
    general_rel_change=0.03,
    fluctuation_threshold=0.08
):
    n = len(group)
    if n < min_points:
        return ["too few points"] * n

    group = group.sort_values('date').reset_index(drop=True)

    dates = group['date']
    values = pd.to_numeric(group[attr], errors="coerce").replace([np.inf, -np.inf], np.nan)

    labels = np.full(n, "no data", dtype=object)

    for i in range(n):
        window_start = dates.iloc[i] - timedelta(days=lookback_days)
        past_mask = (dates >= window_start) & (dates <= dates.iloc[i])

        past_vals = values[past_mask & values.notna()]

        if len(past_vals) < min_points:
            labels[i] = "too few points"
            continue

        v_min   = past_vals.min()
        v_max   = past_vals.max()
        v_first = past_vals.iloc[0]
        v_last  = past_vals.iloc[-1]

        if pd.isna(v_first) or pd.isna(v_last):
            labels[i] = "no data"
            continue

        abs_change = v_last - v_first
        range_pct  = (v_max - v_min) / (abs(v_first) + abs(v_last) + 1e-9) / 2   # rough relative range

        if abs_change <= 1e-6 and range_pct < fluctuation_threshold:
            labels[i] = "stable"
            continue

        rel_change = abs_change / (abs(v_first) + 1e-9)

        is_increasing = abs_change > 0
        is_decreasing = abs_change < 0

        is_strict = abs(rel_change) >= strict_rel_change
        is_general = abs(rel_change) >= general_rel_change

        is_fluctuated = range_pct > fluctuation_threshold

        if is_fluctuated:
            labels[i] = "fluctuated"
        elif is_increasing:
            if is_strict:
                labels[i] = "strictly increasing"
            elif is_general:
                labels[i] = "generally increasing"
            else:
                labels[i] = "slightly increasing"
        elif is_decreasing:
            if is_strict:
                labels[i] = "strictly decreasing"
            elif is_general:
                labels[i] = "generally decreasing"
            else:
                labels[i] = "slightly decreasing"
        else:
            labels[i] = "stable"

    return labels.tolist()

## Computing Trendy Columns (Failure + Attributes)

In [ ]:
print("Computing upcoming failure labels...")
df = df.sort_values(['serial_number', 'date']).reset_index(drop=True)

groups = [group for _, group in df.groupby('serial_number')]
num_groups = len(groups)

num_workers = os.cpu_count()
with Pool(num_workers) as pool:
    upcoming_labels = []
    for labels in tqdm(
        pool.imap(compute_upcoming_vectorized, groups),
        total=num_groups,
        desc="Upcoming failure"
    ):
        upcoming_labels.extend(labels)

df['upcoming_failure'] = upcoming_labels

df["failure_7d"] = (
    df["upcoming_failure"].astype(str).str.startswith("Yes")
    | (pd.to_numeric(df["failure"], errors="coerce").fillna(0).astype(int) == 1)
).astype(int)

In [ ]:
df_fail = df[df['failure'] == 1]
cluster_failures = defaultdict(list)
vault_failures = defaultdict(list)
pod_failures = defaultdict(list)
slot_failures = defaultdict(list)

for _, row in tqdm(df_fail.iterrows(), total=len(df_fail), desc="Building failure timelines"):
    dc = row['datacenter']
    c = int(row['cluster_id']) if pd.notna(row['cluster_id']) else 0
    v = int(row['vault_id']) if pd.notna(row['vault_id']) else 0
    p = int(row['pod_id']) if pd.notna(row['pod_id']) else 0
    serial = row['serial_number']
    dte = pd.to_datetime(row['date'])
    cluster_failures[(dc, c)].append(dte)
    vault_failures[(dc, c, v)].append(dte)
    pod_failures[(dc, c, v, p)].append(dte)
    slot_failures[serial].append(dte)

for d in [cluster_failures, vault_failures, pod_failures, slot_failures]:
    for k in d:
        d[k].sort()

In [ ]:
print("Precomputing recent failure levels...")
df['has_recent_failure'] = False
df['recent_level'] = None

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Recent failures"):
    date_val = row['date']
    dc = row['datacenter']
    c = int(row['cluster_id']) if pd.notna(row['cluster_id']) else 0
    v = int(row['vault_id']) if pd.notna(row['vault_id']) else 0
    p = int(row['pod_id']) if pd.notna(row['pod_id']) else 0
    key_slot = row['serial_number']
    key_pod = (dc, c, v, p)
    key_vault = (dc, c, v)

    levels = [
        ('slot', slot_failures.get(key_slot, [])),
        ('pod', pod_failures.get(key_pod, [])),
        ('vault', vault_failures.get(key_vault, [])),
    ]
    has_recent, recent_level = has_recent_failure(levels, date_val, timedelta(days=7))
    df.at[idx, 'has_recent_failure'] = has_recent
    if recent_level:
        df.at[idx, 'recent_level'] = recent_level

In [ ]:
trendy_attribute_list = df_smart_des.loc[df_smart_des['threshold'] == 'trend', 'name'].tolist()
existing_trendy_attributes = [attr for attr in df.columns if attr.startswith("smart_") and attr in trendy_attribute_list]

In [ ]:
def compute_trend_for_group(group, trendy_col):
    return compute_recent_trend_vectorized(
        group,
        attr=trendy_col,
        lookback_days=7,
        min_points=3,
        strict_rel_change=0.12,
        general_rel_change=0.035,
        fluctuation_threshold=0.085
    )

In [ ]:
for index in range(len(df)):
    if df.iloc[index]['recent_level'] is not None:
        print(index)
        break

In [ ]:
trend_columns = []
groups = [g for _, g in df.groupby('serial_number')]

for trendy_col in existing_trendy_attributes:
    def worker(g):
        return compute_trend_for_group(g, trendy_col)
    
    with Pool() as pool:
        results = list(tqdm(
            pool.imap(worker, groups, chunksize=10),
            total=len(groups),
            desc=f"Computing recent trend for {trendy_col}"
        ))

    all_labels = [label for sublist in results for label in sublist]
    trendy_col_name = f"{trendy_col}_trend_last_7d"
    df[trendy_col_name] = all_labels
    trend_columns.append(trendy_col_name)

## Feature Columns

In [ ]:
feature_columns = [
    col for col in df.columns 
    if col.startswith('smart_') 
    and col in threshold_rules 
    and threshold_rules[col] not in ('info', 'ignore', 'trend', 'model_specific')
]

cols_strong = []
cols_medium = []
for col in feature_columns:
    var_val = df_smart_des.loc[df_smart_des['name'] == col, 'variation']
    if not var_val.empty:
        var = var_val.iloc[0]
        if var == 'Strong':
            cols_strong.append(col)
        elif var == 'Medium':
            cols_medium.append(col)

pod_vault_correlated = set(cols_strong + cols_medium)

## Description Generation

In [ ]:
def generate_single_description(args):
    idx, row, threshold_rules, quantiles, feature_columns, related_map = args
    parts = []

    date_val = pd.to_datetime(row['date'])
    date_str = date_val.strftime('%Y/%m/%d') if pd.notna(date_val) else 'N/A'
    serial = str(row['serial_number']) if pd.notna(row['serial_number']) else 'N/A'
    model = str(row['model']) if pd.notna(row['model']) else 'N/A'
    upcoming = row['upcoming_failure']

    basic_parts = [
        f"Disk: {serial}",
        f"  Date: {date_str}",
        f"  Model: {model}",
        f"  Model: {model}, {query_disk_info(model, 'Manufacturer')}",
        f"  Production Serial: {query_disk_info(model, 'SerialFamily')}",
        f"  Endurance: {query_disk_info(model, 'Endurance')}",
        f"  Failure in next 7 days: {upcoming}"
    ]
    parts.extend(basic_parts)

    has_recent = row['has_recent_failure']
    recent_level = row['recent_level']

    # --- Find abnormal attributes ---
    abnormal_attrs = set()
    attr_descriptions = {}

    for col in feature_columns:
        val = row[col]
        desc = get_description(val, col, threshold_rules, quantiles)
        attr_descriptions[col] = desc
        if desc not in ('normal', 'N/A'):
            abnormal_attrs.add(col)

    # --- Expand to include related attributes ---
    expanded_attrs = set(abnormal_attrs)
    for attr in abnormal_attrs:
        expanded_attrs.update(related_map.get(attr, set()))
    
    # Keep only valid feature columns
    expanded_attrs = expanded_attrs & set(feature_columns)
    expanded_attrs = sorted(expanded_attrs)  # for consistent order

    if not expanded_attrs:
        parts.append("SMART Features: All normal")
    else:
        parts.append("SMART Features:")
        for col in expanded_attrs:
            val = row[col]
            feature_desc = attr_descriptions.get(col, 'N/A')
            if feature_desc == 'N/A':
                feature_desc = get_description(val, col, threshold_rules, quantiles)

            if feature_desc == 'N/A':
                continue
            
            meta_row = df_smart_des[df_smart_des['name'] == col]
            if meta_row.empty:
                feature_meta_des = col
                desired_trend = 'unknown'
            else:
                meta_row = meta_row.iloc[0]
                feature_meta_des = meta_row.get('description', col)
                desired_trend = meta_row.get('desired', 'unknown')

            line = ""
            if desired_trend == 'lower' or desired_trend == 'higher':
                line = f"{col} | {feature_meta_des} | {desired_trend} the better | current status: {feature_desc}"
            else:
                continue
            
            # Add recent failure tag only for originally abnormal attrs
            if has_recent and col in pod_vault_correlated and col in abnormal_attrs:
                line += f" | recent {recent_level} failure"
                
            parts.append(f"  {line}")

    for trend_col in trend_columns:
        trend_val = row[trend_col]
        if trend_val not in ("stable", "too few points", "no data", "fluctuated"):
            base_col = trend_col.replace('_trend_last_7d', '')
            raw_val = row.get(base_col, 'N/A')
            meta_row = df_smart_des[df_smart_des['name'] == base_col]
            if meta_row.empty:
                feature_meta_des = base_col
            else:
                meta_row = meta_row.iloc[0]
                feature_meta_des = meta_row.get('description', col)
            line = f"{base_col} | {feature_meta_des} | 7-Day Trend: {trend_val} "
            parts.append(f"  {line}")

    return idx, '\n'.join(parts)

In [ ]:
def init_worker(threshold_rules, quantiles, feature_columns, related_map):
    global threshold_rules_g, quantiles_g, feature_columns_g, related_map_g
    threshold_rules_g = threshold_rules
    quantiles_g = quantiles
    feature_columns_g = feature_columns
    related_map_g = related_map


def generate_wrapper(pos_and_row):
    pos, row_dict = pos_and_row
    full_args = (pos, row_dict, threshold_rules_g, quantiles_g, feature_columns_g, related_map_g)
    return generate_single_description(full_args)

## Parallel processing

In [ ]:
failed_serials = df[df['failure'] == 1]['serial_number'].unique()
print(f"Number of serials with at least one failure: {len(failed_serials):,}")

all_serials = df['serial_number'].unique()
healthy_serials = np.setdiff1d(all_serials, failed_serials)
print(f"Number of serials that never failed: {len(healthy_serials):,}")

n_sample = int(len(healthy_serials) * (1/2))
sampled_healthy = np.random.choice(healthy_serials, size=n_sample, replace=False)
print(f"Selected {len(sampled_healthy):,} healthy serials (≈1/2)")

keep_serials = np.concatenate([failed_serials, sampled_healthy])
keep_serials_set = set(keep_serials)
df_sampled = df[df['serial_number'].isin(keep_serials_set)].copy()

print(f"Original rows  : {len(df):,}")
print(f"After sampling : {len(df_sampled):,}")
print(f"Reduction      : {len(df) - len(df_sampled):,} rows removed")
print(f"Remaining serials: {df_sampled['serial_number'].nunique():,}")

df = df_sampled

In [ ]:
batch_size = 40000
num_workers = max(1, multiprocessing.cpu_count() - 2)
TEXTUAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

saved_files = []
current_start = df["date"].min()
max_date = df["date"].max()

overall_pbar = tqdm(total=len(df), desc="Overall progress", unit="rows")

with ProcessPoolExecutor(
    max_workers=num_workers,
    initializer=init_worker,
    initargs=(threshold_rules, quantiles, feature_columns, related_map)
) as executor:

    while current_start <= max_date:
        current_end = current_start + pd.DateOffset(months=1)
        mask = (df["date"] >= current_start) & (df["date"] < current_end)
        batch_df = df[mask].copy()

        if batch_df.empty:
            current_start = current_end
            continue

        chunk_start_str = current_start.strftime("%Y-%m-%d")
        chunk_end_str = (current_end - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        chunk_file = TEXTUAL_OUTPUT_DIR / f"backblaze_textual_{chunk_start_str}_to_{chunk_end_str}.csv"

        print(f"\nProcessing chunk: {chunk_start_str} → {chunk_end_str} "
              f"({len(batch_df)} rows) → saving to {chunk_file}")

        records = batch_df.to_dict(orient="records")
        chunk_descriptions = ["Generation failed"] * len(batch_df)

        chunk_pbar = tqdm(total=len(batch_df), desc="Generating descriptions", leave=False)

        for local_start in range(0, len(batch_df), batch_size):
            local_end = min(local_start + batch_size, len(batch_df))
            sub_records = records[local_start:local_end]

            futures = []
            for sub_pos, row_dict in enumerate(sub_records):
                pos = local_start + sub_pos
                future = executor.submit(generate_wrapper, (pos, row_dict))
                futures.append(future)

            for future in as_completed(futures):
                try:
                    idx, desc = future.result()
                    chunk_descriptions[idx] = desc
                except Exception as exc:
                    print(f"Warning: error {exc}")
                    pass  # keep "Generation failed"
                chunk_pbar.update(1)

        chunk_pbar.close()
        overall_pbar.update(len(batch_df))

        new_chunk_df = pd.DataFrame({"description": chunk_descriptions})
        new_chunk_df["failure"] = batch_df["failure"].values
        new_chunk_df["failure_7d"] = batch_df["failure_7d"].values
        new_chunk_df["date"] = batch_df["date"].values
        new_chunk_df["serial_number"] = batch_df["serial_number"].values
        new_chunk_df["model"] = batch_df["model"].values

        new_chunk_df.to_csv(chunk_file, index=False)

        successful_count = (new_chunk_df["description"] != "Generation failed").sum()
        print(f"Saved {len(new_chunk_df)} rows ({successful_count} successful) to {chunk_file}")
        saved_files.append(str(chunk_file))

        del batch_df, records, chunk_descriptions, new_chunk_df
        gc.collect()

        current_start = current_end

overall_pbar.close()
print("All chunks processed.")

In [ ]:
d2t_end_time = time.perf_counter()
elapsed_time = d2t_end_time - d2t_start_time
print(f"D2T elapsed time: {elapsed_time:.6f} seconds")